In [ ]:
import sys
import torch
import numpy as np
import rasterio
import terratorch

print("Python 路径:", sys.executable)
print("PyTorch 版本:", torch.__version__)
print("CUDA 是否可用:", torch.cuda.is_available())
print("CUDA 版本:", torch.version.cuda)
print("numpy 版本:", np.__version__)
print("rasterio 导入成功")
print("terratorch 导入成功")

In [ ]:
import torch
from terratorch.registry import BACKBONE_REGISTRY

# 自动选择 GPU 或 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 你当前能成功加载的 Prithvi-EO-2.0 注册名
model_name = "prithvi_eo_v2_300"

print("当前设备:", device)
print("开始加载模型:", model_name)

# 加载预训练 Prithvi backbone
backbone = BACKBONE_REGISTRY.build(model_name, pretrained=True)

# 移动到 GPU / CPU
backbone = backbone.to(device)

# 设置为推理模式
backbone.eval()

print("Prithvi-EO-2.0 加载成功")
print(type(backbone))

In [ ]:
# 冻结 Prithvi backbone，后面只训练任务头
for param in backbone.parameters():
    param.requires_grad = False

# 再次设置为 eval，防止 BatchNorm / Dropout 等层发生训练行为
backbone.eval()

# 检查可训练参数数量
total_params = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)

print("Prithvi 总参数量:", total_params)
print("Prithvi 可训练参数量:", trainable_params)

In [ ]:
# 构造一个假的 6 波段 224×224 输入
# 形状：[batch, channels, height, width]
x = torch.randn(1, 6, 224, 224).to(device)

with torch.no_grad():
    out = backbone(x)

print("输出类型:", type(out))

# 查看不同输出结构
if isinstance(out, torch.Tensor):
    print("输出 shape:", out.shape)

elif isinstance(out, (list, tuple)):
    print("输出数量:", len(out))
    for i, item in enumerate(out):
        if isinstance(item, torch.Tensor):
            print(f"第 {i} 个输出 shape:", item.shape)
        else:
            print(f"第 {i} 个输出类型:", type(item))

elif isinstance(out, dict):
    print("输出 keys:", out.keys())
    for k, v in out.items():
        if isinstance(v, torch.Tensor):
            print(k, v.shape)
        else:
            print(k, type(v))

In [ ]:
def extract_feature_from_output(out):
    """
    从 Prithvi 输出中提取一个 tensor 特征。
    兼容 tensor / list / tuple / dict 等不同输出格式。
    """
    
    # 情况1：直接输出 tensor
    if isinstance(out, torch.Tensor):
        feat = out

    # 情况2：输出 list 或 tuple，优先取最后一个 tensor
    elif isinstance(out, (list, tuple)):
        tensor_items = [item for item in out if isinstance(item, torch.Tensor)]
        if len(tensor_items) == 0:
            raise ValueError("Prithvi 输出 list/tuple 中没有 tensor")
        feat = tensor_items[-1]

    # 情况3：输出 dict，优先找常见 key
    elif isinstance(out, dict):
        possible_keys = ["last_hidden_state", "features", "out", "embedding", "x"]
        feat = None

        for key in possible_keys:
            if key in out and isinstance(out[key], torch.Tensor):
                feat = out[key]
                break

        # 如果常见 key 找不到，就取第一个 tensor value
        if feat is None:
            for value in out.values():
                if isinstance(value, torch.Tensor):
                    feat = value
                    break

        if feat is None:
            raise ValueError("Prithvi 输出 dict 中没有 tensor")

    else:
        raise TypeError(f"暂不支持的输出类型: {type(out)}")

    return feat


def pool_feature(feat):
    """
    将 Prithvi 的空间/token 特征池化成 patch-level 向量。
    输出形状尽量变成 [batch, feature_dim]。
    """

    # 如果已经是 [B, D]
    if feat.ndim == 2:
        return feat

    # 如果是 [B, N, D]，对 token 维度求平均
    if feat.ndim == 3:
        return feat.mean(dim=1)

    # 如果是 [B, C, H, W]，做全局平均池化
    if feat.ndim == 4:
        return feat.mean(dim=(2, 3))

    # 如果是 [B, T, C, H, W]，对 T/H/W 求平均
    if feat.ndim == 5:
        return feat.mean(dim=(1, 3, 4))

    raise ValueError(f"暂不支持的特征维度: {feat.shape}")

In [ ]:
# 这里默认先用 4D 输入，如果你的 Cell 4 报错，就改成 5D 输入
x = torch.randn(1, 6, 224, 224).to(device)

with torch.no_grad():
    out = backbone(x)
    feat = extract_feature_from_output(out)
    pooled_feat = pool_feature(feat)

print("原始特征 shape:", feat.shape)
print("池化后特征 shape:", pooled_feat.shape)